## PHASE 3
### Transform raw tables into useful business data

In [133]:
import pandas as pd

In [134]:
orders = pd.read_csv("../data/processed/orders_cleaned.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

#parse_dates=[]
#It tells Pandas:
#"These columns contain dates. Convert them into Pandas datetime format when reading the file."
#Without parse_dates, Pandas may read them as strings (object).
#With parse_dates, Pandas converts it into a proper datetime value.

In [135]:
orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
delivery_days                           float64
delivery_delay_days                     float64
dtype: object

In [136]:
order_items = pd.read_csv(
    "../data/processed/order_items_cleaned.csv",
    parse_dates=["shipping_limit_date"]
)

products = pd.read_csv(
    "../data/processed/products_cleaned.csv"
)

customers = pd.read_csv(
    "../data/processed/customers_cleaned.csv"
)

sellers = pd.read_csv(
    "../data/processed/sellers_cleaned.csv"
)

payments = pd.read_csv(
    "../data/processed/payments_cleaned.csv"
)

reviews = pd.read_csv(
    "../data/processed/reviews_cleaned.csv",
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp"
    ]
)

category_translation = pd.read_csv(
    "../data/processed/category_translation_cleaned.csv"
)

In [137]:
orders[["order_id", "customer_id", "order_status"]].head()

,order_id,customer_id,order_status
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered


In [138]:
order_items[
    ["order_id", "order_item_id", "product_id", "price"]
].head()

,order_id,order_item_id,product_id,price
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,58.90
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,239.90
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,199.00
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,199.90


In [139]:
order_item_data=orders.merge(
    order_items,
    on="order_id",
    how="left")

order_item_data.head()

#left join: Keep every order, and attach order-item information where it exists.

#keeps all orders even if it has no matching in order_item

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72


In [140]:
order_item_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 16 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113425 non-null  str           
 1   customer_id                    113425 non-null  str           
 2   order_status                   113425 non-null  str           
 3   order_purchase_timestamp       113425 non-null  datetime64[us]
 4   order_approved_at              113264 non-null  datetime64[us]
 5   order_delivered_carrier_date   111457 non-null  datetime64[us]
 6   order_delivered_customer_date  110196 non-null  datetime64[us]
 7   order_estimated_delivery_date  113425 non-null  datetime64[us]
 8   delivery_days                  110196 non-null  float64       
 9   delivery_delay_days            110196 non-null  float64       
 10  order_item_id                  112650 non-null  float64       
 11  product_id 

In [141]:
orders.shape

(99441, 10)

In [142]:
#This should be larger than the number of orders because one order can have multiple order items.
order_item_data.shape

(113425, 16)

This is where your earlier understanding of grain becomes extremely important.

Before:

orders
1 row ≈ 1 order

After:

order_item_data
1 row ≈ 1 order item

That means we've changed the grain.

## Merge product 

Suppose both DataFrames have a column city.
When you merge them, Pandas finds that city exists in both DataFrames.
So it needs different names for the two city columns.

suffixes=("", "_product")

The first suffix:""

means:
Don't add anything to the column coming from order_item_data.

The second suffix:"_product"

means:
Add _product to duplicate columns coming from products.

In [143]:
order_item_data= order_item_data.merge(
    products,
    on="product_id",
    how="left",
    suffixes=("","_product")
)
order_item_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 24 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113425 non-null  str           
 1   customer_id                    113425 non-null  str           
 2   order_status                   113425 non-null  str           
 3   order_purchase_timestamp       113425 non-null  datetime64[us]
 4   order_approved_at              113264 non-null  datetime64[us]
 5   order_delivered_carrier_date   111457 non-null  datetime64[us]
 6   order_delivered_customer_date  110196 non-null  datetime64[us]
 7   order_estimated_delivery_date  113425 non-null  datetime64[us]
 8   delivery_days                  110196 non-null  float64       
 9   delivery_delay_days            110196 non-null  float64       
 10  order_item_id                  112650 non-null  float64       
 11  product_id 

In [144]:
order_item_data = order_item_data.merge(
    sellers,
    on="seller_id",
    how="left",
    suffixes=("", "_seller")
)
order_item_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 27 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113425 non-null  str           
 1   customer_id                    113425 non-null  str           
 2   order_status                   113425 non-null  str           
 3   order_purchase_timestamp       113425 non-null  datetime64[us]
 4   order_approved_at              113264 non-null  datetime64[us]
 5   order_delivered_carrier_date   111457 non-null  datetime64[us]
 6   order_delivered_customer_date  110196 non-null  datetime64[us]
 7   order_estimated_delivery_date  113425 non-null  datetime64[us]
 8   delivery_days                  110196 non-null  float64       
 9   delivery_delay_days            110196 non-null  float64       
 10  order_item_id                  112650 non-null  float64       
 11  product_id 

In [145]:
order_item_data = order_item_data.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

In [146]:
order_item_data.shape

(113425, 28)

In [147]:
order_item_data.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,...,268.0,4.0,500.0,19.0,8.0,13.0,9350.0,maua,SP,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,...,178.0,1.0,400.0,19.0,13.0,19.0,31570.0,belo horizonte,SP,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,...,232.0,1.0,420.0,24.0,19.0,21.0,14840.0,guariba,SP,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,...,468.0,3.0,450.0,30.0,10.0,20.0,31842.0,belo horizonte,MG,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,...,316.0,4.0,250.0,51.0,15.0,15.0,8752.0,mogi das cruzes,SP,stationery


In [148]:
order_item_data.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_days',
 'delivery_delay_days',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state',
 'product_category_name_english']

Check whether the joins worked

In [149]:
order_item_data["product_id"].isna().sum()

np.int64(775)

In [150]:
order_item_data["seller_id"].isna().sum()

np.int64(775)

In [151]:
order_item_data["product_category_name_english"].isna().sum()

np.int64(2402)

In [152]:
order_item_data["item_total"] = (
    order_item_data["price"]
    + order_item_data["freight_value"]
)

In [153]:
order_item_data[
    ["price", "freight_value", "item_total"]
].head()

,price,freight_value,item_total
0,29.99,8.72,38.71
1,118.70,22.76,141.46
2,159.90,19.22,179.12
3,45.00,27.20,72.20
4,19.90,8.72,28.62


In [154]:
#applying different aggregate function to different columns

order_summary=(order_item_data.groupby("order_id",as_index=False)).agg(
    order_item_revenue=("price","sum"),
    total_freight=("freight_value","sum"),
    total_order_value=("item_total",sum),
    item_count=("order_id","count")
)

order_summary.sort_values("item_count",ascending=False)


,order_id,order_item_revenue,total_freight,total_order_value,item_count
50543,8272b63d03f5f79c56e9e4120aec44ef,31.80,164.37,196.17,21
10541,1b15974a0141d54e36626dca3fdc731a,2000.00,202.40,2202.40,20
66248,ab14fdcfbe524636d65ee38360e22ce8,1974.00,288.80,2262.80,20
25797,428a2f660dc84138d969ccd69a0ab6d5,982.35,243.30,1225.65,15
61436,9ef13efd6949e4573a18964dd1bbe7f5,765.00,18.00,783.00,15
...,...,...,...,...,...
99420,fff1e3e76b816bfe8ef16678cc53c643,65.99,20.86,86.85,1
99421,fff2cdc825f9fc0ba3c04227cfa02303,24.99,25.63,50.62,1
99422,fff2e9e3aa8644e19710216b4ef53ab2,69.90,16.25,86.15,1
99423,fff3983dfa3c5a0d752d8d17baa406a0,66.39,14.05,80.44,1


In [155]:
order_summary.shape

(99441, 5)

In [156]:
order_summary["item_count"].describe()

count    99441.000000
mean         1.140626
std          0.536495
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: item_count, dtype: float64

### Join customers

In [157]:
order_item_data=order_item_data.merge(
    customers,
    on="customer_id",
    how="left",
    suffixes=("","_customer")
)

In [158]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [159]:
payments.columns.tolist()

['order_id',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value']

In [160]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [161]:
payments.groupby("order_id").size().describe()

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64

For example:

Order A
│
├── Credit card → ₹100
└── Voucher     → ₹50

becomes:

Order A
payment_value = ₹150
payment_count = 2

We have compressed multiple payment records into one order-level record.

In [162]:
payment_summary=(payments.groupby("order_id",as_index=False).agg(
    payment_value=("payment_value","sum"),
    payment_count=("payment_sequential","count"),
    payment_installments_max=("payment_installments","max")
))

payment_summary.head()

,order_id,payment_value,payment_count,payment_installments_max
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3


In [163]:
order_item_data=order_item_data.merge(
    payment_summary,
    on="order_id",
    how="left"
)

Add Review Information

In [164]:
# size() counts the number of rows in each group.
# describe() gives statistical information about those counts.
reviews.groupby("order_id").size().describe()

count    98673.000000
mean         1.005584
std          0.075060
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000
dtype: float64

In [165]:
reviews.columns.tolist()

['review_id',
 'order_id',
 'review_score',
 'review_comment_title',
 'review_comment_message',
 'review_creation_date',
 'review_answer_timestamp']

In [166]:
review_summary=(reviews.groupby("order_id",as_index=False).agg(
    review_score=("review_score","mean"),
    review_count=("review_id","nunique")
))

review_summary.head()

,order_id,review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


In [167]:
order_item_data=order_item_data.merge(
    review_summary,
    on="order_id",
    how="left"
)

In [168]:
def classify_review(score):
    if pd.isna(score):
        return "No review"
    elif score<=2:
        return "poor"
    elif score == 3:
        return "average"
    else:
        return "good"

classify_review(2)

'poor'

In [169]:
order_item_data["review_category"]=(order_item_data["review_score"].apply(classify_review))
order_item_data["review_category"].value_counts()

review_category
good         84497
poor         18530
average       9437
No review      961
Name: count, dtype: int64

Add delivery performance

In [170]:
order_item_data[
    [
        "order_id",
        "delivery_days",
        "delivery_delay_days"
    ]
].head()

,order_id,delivery_days,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,8.0,-8.0
1,53cdb2fc8bc7dce0b6741e2150273451,13.0,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,9.0,-18.0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0,-13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,2.0,-10.0


In [171]:
order_item_data["delivery_status"]="On Time"

In [172]:
order_item_data.loc[
    order_item_data["delivery_delay_days"]>0,
    "delivery_status"
]="Late"

In [173]:
order_item_data.loc[
    order_item_data["delivery_delay_days"].isna(),
    "delivery_status"
]="Not Delivered"

In [174]:
order_item_data["delivery_status"].value_counts()

delivery_status
On Time          102931
Late               7265
Not Delivered      3229
Name: count, dtype: int64

In [175]:
order_item_data.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_days',
 'delivery_delay_days',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state',
 'product_category_name_english',
 'item_total',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'payment_value',
 'payment_count',
 'payment_installments_max',
 'review_score',
 'review_count',
 'review_category',
 'delivery_status']

In [176]:
print("orders:", orders["order_id"].nunique())
print("order_item_data rows:", len(order_item_data))
print(
    "unique order-item combinations:",
    order_item_data[["order_id", "order_item_id"]]
    .drop_duplicates()
    .shape[0]
)

orders: 99441
order_item_data rows: 113425
unique order-item combinations: 113425


Validate the analytical join

In [177]:
print("Original order_items rows:", len(order_items))

print(
    "Unique order-item combinations:",
    order_item_data[["order_id", "order_item_id"]]
    .drop_duplicates()
    .shape[0]
)

print(
    "Current order_item_data rows:",
    len(order_item_data)
)

Original order_items rows: 112650
Unique order-item combinations: 113425
Current order_item_data rows: 113425


Check for accidental duplication

In [178]:
#"Check whether the same order contains the same item ID more than once, and count how many duplicate occurrences exist."
duplicate_item_rows = order_item_data.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

duplicate_item_rows

np.int64(0)

Check the financial numbers

Before combining tables

In [179]:
original_item_revenue = order_items["price"].sum()
original_freight = order_items["freight_value"].sum()

print("Original item revenue:", original_item_revenue)
print("Original freight:", original_freight)

Original item revenue: 13591643.7
Original freight: 2251909.54


After combining tables

In [180]:
joined_item_revenue = order_item_data["price"].sum()
joined_freight = order_item_data["freight_value"].sum()

print("Joined item revenue:", joined_item_revenue)
print("Joined freight:", joined_freight)

#These should match.
#Why?
#Because joining product/customer/seller information shouldn't magically create additional products or money.
#If they differ, we've got a join problem.

Joined item revenue: 13591643.7
Joined freight: 2251909.54


Verify our item_total

In [181]:
order_item_data["item_total"] = (
    order_item_data["price"]
    + order_item_data["freight_value"]
)

In [182]:
print("Calculated item total:",
      order_item_data["item_total"].sum()
)

Calculated item total: 15843553.24


In [183]:
print("Expected item total:",
    original_item_revenue + original_freight
)

Expected item total: 15843553.239999998


Validate payment aggregation

In [184]:
payment_summary

,order_id,payment_value,payment_count,payment_installments_max
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3
...,...,...,...,...
99435,fffc94f6ce00a00581880bf54a75a037,343.40,1,1
99436,fffcd46ef2263f404302a634eb57f7eb,386.53,1,1
99437,fffce4705a9662cd70adb13d4a31832d,116.85,1,3
99438,fffe18544ffabc95dfada21779c9644f,64.71,1,3


In [185]:
print(
    "Original payment total:",
    payments["payment_value"].sum()
)

print(
    "Aggregated payment total:",
    payment_summary["payment_value"].sum()
)

Original payment total: 16008872.120000001
Aggregated payment total: 16008872.12


These numbers don't need to be equal.

Why?

Because:

reviews
1 row = review

while:

review_summary
1 row = order

We're intentionally changing the grain.

In [186]:
print(
    "Original review count:",
    reviews["review_id"].nunique()
)

print(
    "Orders with reviews:",
    review_summary["order_id"].nunique()
)

Original review count: 98410
Orders with reviews: 98673


## Analytical Model Validation

The analytical dataset is designed at the order-item grain:

> One row represents one product line within an order.

Validation checks confirm that:

- Order-item row counts are preserved after joins.
- `(order_id, order_item_id)` remains unique.
- Product, seller, and order references remain valid.
- Item revenue is unchanged by dimensional joins.
- Freight totals are unchanged by dimensional joins.
- Payment totals are preserved during payment aggregation.
- Review information is aggregated to the order level before joining.

These checks protect against accidental row multiplication and incorrect business metrics.

Product dimension

In [187]:
dim_products = products.copy()

In [188]:
dim_products=dim_products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

dim_products.shape

(32951, 10)

In [189]:
dim_products["product_id"].duplicated().sum()

np.int64(0)

customer dimension

In [190]:
dim_customers=customers.copy()

In [191]:
dim_customers.shape

(99441, 5)

In [192]:
dim_customers["customer_id"].duplicated().sum()

np.int64(0)

seller dimension

In [193]:
dim_sellers=sellers.copy()

In [194]:
dim_sellers["seller_id"].duplicated().sum()

np.int64(0)

In [195]:
print("Original order_items:", len(order_items))
print(
    "Unique order-item pairs:",
    order_item_data[["order_id", "order_item_id"]]
    .drop_duplicates()
    .shape[0]
)
print("Current order_item_data:", len(order_item_data))
print(
    "Duplicate order-item pairs:",
    order_item_data.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Original order_items: 112650
Unique order-item pairs: 113425
Current order_item_data: 113425
Duplicate order-item pairs: 0


In [196]:
print(
    "Original item revenue:",
    order_items["price"].sum()
)
print(
    "Joined item revenue:",
    order_item_data["price"].sum()
)

Original item revenue: 13591643.7
Joined item revenue: 13591643.7


## Build the analytical model

dim_products table

In [197]:
#But explicitly selecting columns:
#ensures only the required columns remain
#It also controls column order

dim_products = dim_products[
    [
        "product_id",
        "product_category_name",
        "product_category_name_english",
        "product_name_lenght",
        "product_description_lenght",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
]

In [198]:
print("Rows:", len(dim_products))
print("Unique products:", dim_products["product_id"].nunique())
print(
    "Duplicate product IDs:",
    dim_products["product_id"].duplicated().sum()
)

Rows: 32951
Unique products: 32951
Duplicate product IDs: 0


dim_customers table

In [199]:
dim_customers = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]
].copy()

In [200]:
print("Rows:",len(dim_customers))
print("Unique customers IDs:",dim_customers["customer_id"].nunique())
print("Duplicated rows:",dim_customers["customer_id"].duplicated().sum())

Rows: 99441
Unique customers IDs: 99441
Duplicated rows: 0


dim_sellers table

In [201]:
dim_sellers = sellers[
    [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ]
].copy()

In [202]:
print("Rows:", len(dim_sellers))
print(
    "Unique seller IDs:",
    dim_sellers["seller_id"].nunique()
)
print(
    "Duplicate seller IDs:",
    dim_sellers["seller_id"].duplicated().sum()
)

Rows: 3095
Unique seller IDs: 3095
Duplicate seller IDs: 0


fact_orders table

In [203]:
orders["delivery_status"] = "On Time"

orders.loc[
    orders["delivery_delay_days"] > 0,
    "delivery_status"
] = "Late"

orders.loc[
    orders["delivery_delay_days"].isna(),
    "delivery_status"
] = "Not Delivered"

We referenced delivery_status as if it already existed in orders, but we actually created it later on order_item_data.

In [204]:
fact_orders = orders[
    [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "delivery_delay_days",
        "delivery_status"
    ]
].copy()

fact_order_items table

In [205]:
fact_order_items = order_item_data[
    [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "customer_id",
        "shipping_limit_date",
        "price",
        "freight_value",
        "item_total"
    ]
].copy()

In [206]:
print("Rows:", len(fact_order_items))
print(
    "Unique order-item pairs:",
    fact_order_items[
        ["order_id", "order_item_id"]
    ].drop_duplicates().shape[0]
)
print(
    "Duplicate order-item pairs:",
    fact_order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Rows: 113425
Unique order-item pairs: 113425
Duplicate order-item pairs: 0


fact_payments table

In [207]:
#We already aggregated payments:
#payment_summary

fact_payments = payment_summary.copy()

fact_reviews table

In [208]:
fact_reviews = review_summary.copy()

Create a model summary

In [209]:
model_summary = {
    "dim_products_rows": len(dim_products),
    "dim_customers_rows": len(dim_customers),
    "dim_sellers_rows": len(dim_sellers),
    "fact_orders_rows": len(fact_orders),
    "fact_order_items_rows": len(fact_order_items),
    "fact_payments_rows": len(fact_payments),
    "fact_reviews_rows": len(fact_reviews),
    "duplicate_order_ids": fact_orders["order_id"].duplicated().sum(),
    "duplicate_order_item_keys": fact_order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
}

model_summary

{'dim_products_rows': 32951,
 'dim_customers_rows': 99441,
 'dim_sellers_rows': 3095,
 'fact_orders_rows': 99441,
 'fact_order_items_rows': 113425,
 'fact_payments_rows': 99440,
 'fact_reviews_rows': 98673,
 'duplicate_order_ids': np.int64(0),
 'duplicate_order_item_keys': np.int64(0)}

Save the model tables

In [210]:
dim_products.to_csv(
    "../data/final/dim_products.csv",
    index=False
)

dim_customers.to_csv(
    "../data/final/dim_customers.csv",
    index=False
)

dim_sellers.to_csv(
    "../data/final/dim_sellers.csv",
    index=False
)

fact_orders.to_csv(
    "../data/final/fact_orders.csv",
    index=False
)

fact_order_items.to_csv(
    "../data/final/fact_order_items.csv",
    index=False
)

fact_payments.to_csv(
    "../data/final/fact_payments.csv",
    index=False
)

fact_reviews.to_csv(
    "../data/final/fact_reviews.csv",
    index=False
)

## Final Analytical Model

The project uses a star-schema-inspired analytical model.

### Dimension tables

- `dim_products` — one row per product
- `dim_customers` — one row per customer record
- `dim_sellers` — one row per seller

### Fact tables

- `fact_orders` — one row per order
- `fact_order_items` — one row per order item
- `fact_payments` — one row per order with aggregated payment information
- `fact_reviews` — one row per order with aggregated review information

### Key design principle

Each table has a clearly defined grain.

This prevents accidental row multiplication and helps ensure that business metrics such as revenue, order count, payment value, and review scores are calculated correctly.

## Create one final validation report

In [211]:
final_model_validation = {
    "order_count_matches": (
        orders["order_id"].nunique()
        == fact_orders["order_id"].nunique()
    ),

    "order_item_count_matches": (
        len(order_items)
        == len(fact_order_items)
    ),

    "order_item_keys_unique": (
        fact_order_items.duplicated(
            subset=["order_id", "order_item_id"]
        ).sum()
        == 0
    ),

    "revenue_reconciles": (
        abs(
            order_items["price"].sum()
            - fact_order_items["price"].sum()
        ) < 0.01
    ),

    "freight_reconciles": (
        abs(
            order_items["freight_value"].sum()
            - fact_order_items["freight_value"].sum()
        ) < 0.01
    ),

    "payments_reconcile": (
        abs(
            payments["payment_value"].sum()
            - fact_payments["payment_value"].sum()
        ) < 0.01
    ),

    "product_references_valid": (
        (~fact_order_items["product_id"].isin(
            dim_products["product_id"]
        )).sum() == 0
    ),

    "seller_references_valid": (
        (~fact_order_items["seller_id"].isin(
            dim_sellers["seller_id"]
        )).sum() == 0
    ),

    "customer_references_valid": (
        (~fact_order_items["customer_id"].isin(
            dim_customers["customer_id"]
        )).sum() == 0
    ),

    "order_references_valid": (
        (~fact_order_items["order_id"].isin(
            fact_orders["order_id"]
        )).sum() == 0
    )
}

final_model_validation

{'order_count_matches': True,
 'order_item_count_matches': False,
 'order_item_keys_unique': np.True_,
 'revenue_reconciles': np.True_,
 'freight_reconciles': np.True_,
 'payments_reconcile': np.True_,
 'product_references_valid': np.False_,
 'seller_references_valid': np.False_,
 'customer_references_valid': np.True_,
 'order_references_valid': np.True_}

In [212]:
final_model_validation

{'order_count_matches': True,
 'order_item_count_matches': False,
 'order_item_keys_unique': np.True_,
 'revenue_reconciles': np.True_,
 'freight_reconciles': np.True_,
 'payments_reconcile': np.True_,
 'product_references_valid': np.False_,
 'seller_references_valid': np.False_,
 'customer_references_valid': np.True_,
 'order_references_valid': np.True_}

In [213]:
all(final_model_validation.values())

False

# error correction

In [214]:
print("Original order items:", len(order_items))
print("Fact order items:", len(fact_order_items))

Original order items: 112650
Fact order items: 113425


In [215]:
missing_product_rows = fact_order_items[
    ~fact_order_items["product_id"].isin(
        dim_products["product_id"]
    )
]

print("Missing product references:", len(missing_product_rows))

Missing product references: 775


In [216]:
missing_product_rows[
    ["order_id", "order_item_id", "product_id"]
].head(20)

,order_id,order_item_id,product_id
306,8e24261a7e58791d10cb1bf9da94df5c,NaN,NaN
671,c272bcd21c287498b4883c7512019702,NaN,NaN
791,37553832a3a89c9b2db59701c357ca67,NaN,NaN
850,d57e15fb07fd180f06ab3926b39edcd2,NaN,NaN
1294,00b1cb0320190ca0daa2c88b35206009,NaN,NaN
1326,2f634e2cebf8c0283e7ef0989f77d217,NaN,NaN
1802,ee0db22a8e742b752914016708470ec8,NaN,NaN
2040,ed3efbd3a87bea76c2812c66a0b32219,NaN,NaN
2074,6ad57aecbae806a7e9cc2cdb6b380711,NaN,NaN
2121,df8282afe61008dc26c6c31011474d02,NaN,NaN


Step 1 — Check products

In [217]:
print("dups product ids:", products["product_id"].duplicated().sum())

print("Duplicate product IDs in dim:",
    dim_products["product_id"].duplicated().sum())

dups product ids: 0
Duplicate product IDs in dim: 0


Step 2 — Check sellers

In [218]:
print("Duplicate seller IDs:",
    sellers["seller_id"].duplicated().sum())

print("Duplicate seller IDs in dim:",
    dim_sellers["seller_id"].duplicated().sum())

Duplicate seller IDs: 0
Duplicate seller IDs in dim: 0


Step 3 — Check customers

In [219]:
print("Duplicate customer IDs:",
    customers["customer_id"].duplicated().sum())

Duplicate customer IDs: 0


Step 4 — Check the category translation table

In [220]:
print("Duplicate category names:",
    category_translation["product_category_name"].duplicated().sum())

Duplicate category names: 0


Step 5 — Find which products are responsible

In [221]:
product_counts = products["product_id"].value_counts()

product_counts[product_counts>1].head(20)

Series([], Name: count, dtype: int64)

Step 6 — Find which sellers are responsible

In [222]:
seller_counts = sellers["seller_id"].value_counts()

seller_counts[seller_counts>1].head(20)

Series([], Name: count, dtype: int64)

Step 7 — Find exactly where the multiplication happened

In [223]:
test_data = orders.merge(
order_items,
    on="order_id",
    how="left"
)

print("After orders + order_items:", len(test_data))

After orders + order_items: 113425


In [224]:
print(len(orders))
print(len(order_items))

99441
112650


In [225]:
orders["order_id"].duplicated().sum()

np.int64(0)

In [226]:
orders["order_id"].nunique()

99441

In [227]:
duplicate_orders = orders[
    orders["order_id"].duplicated(keep=False)
].sort_values("order_id")

duplicate_orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,delivery_status


In [228]:
duplicate_order_ids = (
    orders["order_id"]
    .value_counts()
)

duplicate_order_ids[
    duplicate_order_ids > 1
].head(20)

Series([], Name: count, dtype: int64)

In [229]:
print("orders:", orders.shape)
print("order_items:", order_items.shape)

orders: (99441, 11)
order_items: (112650, 7)


Step 4 — Check whether the JOIN itself is behaving strangely

In [230]:
test_data = orders.merge(
    order_items,
    on="order_id",
    how="left",
    validate="one_to_many"
)
len(test_data)

113425

In [231]:
orders_without_items = orders[
    ~orders["order_id"].isin(order_items["order_id"])
]
print("Orders without order items:", len(orders_without_items))

#There are 775 orders in orders whose order_id does not appear anywhere in order_items.

Orders without order items: 775


In [232]:
orders_without_items["order_status"].value_counts()

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

# Data integrity check:
# 775 orders do not have corresponding order-item records.
# Most are unavailable (603) or canceled (164).
# This is expected for orders that were unavailable/canceled.

❌ What we did before
orders.merge(order_items)

Grain starts as:

orders
1 row = order

✅ What we're doing now
order_items.merge(orders)

Grain starts as:

order_items
1 row = order item

That's exactly what we need.

Yes — you should overwrite it, not worry about deleting it manually.

order_item_data is just a Python variable in your notebook's memory. It is not a file unless you explicitly saved it somewhere

In [233]:
order_item_data = order_items.merge(
    orders,
    on="order_id",
    how="left",
    validate="many_to_one"
)

In [234]:
print(len(order_items))
print(len(order_item_data))

112650
112650


In [235]:
order_item_data.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'customer_id',
       'order_status', 'order_purchase_timestamp', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'delivery_days', 'delivery_delay_days',
       'delivery_status'],
      dtype='str')

It means Pandas will stop the pipeline with an error if, for example, one product_id unexpectedly matches multiple product records.

In [236]:
order_item_data = order_item_data.merge(
    products,
    on="product_id",
    how="left",
    suffixes=("", "_product"),
    validate="many_to_one"
)
print("After products:", len(order_item_data))

After products: 112650


In [237]:
order_item_data = order_item_data.merge(
    sellers,
    on="seller_id",
    how="left",
    suffixes=("", "_seller"),
    validate="many_to_one"
)
print("After sellers:", len(order_item_data))

After sellers: 112650


In [238]:
order_item_data = order_item_data.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)
print("After category translation:", len(order_item_data))

After category translation: 112650


In [239]:
order_item_data = order_item_data.merge(
    customers,
    on="customer_id",
    how="left",
    suffixes=("", "_customer"),
    validate="many_to_one"
)
print("After customers:", len(order_item_data))

After customers: 112650


In [240]:
order_item_data = order_item_data.merge(
    payment_summary,
    on="order_id",
    how="left",
    validate="many_to_one"
)
print("After payments:", len(order_item_data))

After payments: 112650


In [241]:
order_item_data = order_item_data.merge(
    review_summary,
    on="order_id",
    how="left",
    validate="many_to_one"
)
print("After reviews:", len(order_item_data))

After reviews: 112650


Recreate item_total

In [242]:
order_item_data["item_total"] = (
    order_item_data["price"]
    + order_item_data["freight_value"]
)

Recreate review category

In [243]:
def classify_review(score):
    if pd.isna(score):
        return "No Review"
    elif score <= 2:
        return "Poor"
    elif score == 3:
        return "Average"
    else:
        return "Good"

In [244]:
order_item_data["review_category"] = (
    order_item_data["review_score"]
    .apply(classify_review)
)

In [245]:
print("Final order_item_data rows:", len(order_item_data))

print(
    "Duplicate order-item pairs:",
    order_item_data.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Final order_item_data rows: 112650
Duplicate order-item pairs: 0


In [246]:
fact_order_items = order_item_data[
    [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "customer_id",
        "shipping_limit_date",
        "price",
        "freight_value",
        "item_total"
    ]
].copy()

In [247]:
print("Original order_items:", len(order_items))
print("Fact order_items:", len(fact_order_items))

print(
    "Duplicate order-item pairs:",
    fact_order_items.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Original order_items: 112650
Fact order_items: 112650
Duplicate order-item pairs: 0


In [249]:
final_model_validation = {
    "order_count_matches": (
        orders["order_id"].nunique()
        == fact_orders["order_id"].nunique()
    ),

    "order_item_count_matches": (
        len(order_items)
        == len(fact_order_items)
    ),

    "order_item_keys_unique": (
        fact_order_items.duplicated(
            subset=["order_id", "order_item_id"]
        ).sum()
        == 0
    ),

    "revenue_reconciles": (
        abs(
            order_items["price"].sum()
            - fact_order_items["price"].sum()
        ) < 0.01
    ),

    "freight_reconciles": (
        abs(
            order_items["freight_value"].sum()
            - fact_order_items["freight_value"].sum()
        ) < 0.01
    ),

    "payments_reconcile": (
        abs(
            payments["payment_value"].sum()
            - fact_payments["payment_value"].sum()
        ) < 0.01
    ),

    # "product_references_valid": (
    #     missing_products == 0
    # ),

    # "seller_references_valid": (
    #     missing_sellers == 0
    # ),

    # "customer_references_valid": (
    #     missing_customers == 0
    # ),

    # "order_references_valid": (
    #     missing_orders == 0
    # )
}

In [250]:
final_model_validation

{'order_count_matches': True,
 'order_item_count_matches': True,
 'order_item_keys_unique': np.True_,
 'revenue_reconciles': np.True_,
 'freight_reconciles': np.True_,
 'payments_reconcile': np.True_}

In [251]:
all(final_model_validation.values())

True

Phase 3 — Final Save & GitHub Checkpoint

In [252]:
dim_products.to_csv(
    "../data/final/dim_products.csv",
    index=False
)

dim_customers.to_csv(
    "../data/final/dim_customers.csv",
    index=False
)

dim_sellers.to_csv(
    "../data/final/dim_sellers.csv",
    index=False
)

fact_orders.to_csv(
    "../data/final/fact_orders.csv",
    index=False
)

fact_order_items.to_csv(
    "../data/final/fact_order_items.csv",
    index=False
)

fact_payments.to_csv(
    "../data/final/fact_payments.csv",
    index=False
)

fact_reviews.to_csv(
    "../data/final/fact_reviews.csv",
    index=False
)

print("Final analytical tables saved successfully.")

Final analytical tables saved successfully.


In [253]:
print(
    "fact_orders:",
    pd.read_csv("../data/final/fact_orders.csv").shape
)

print(
    "fact_order_items:",
    pd.read_csv("../data/final/fact_order_items.csv").shape
)

print(
    "fact_payments:",
    pd.read_csv("../data/final/fact_payments.csv").shape
)

print(
    "fact_reviews:",
    pd.read_csv("../data/final/fact_reviews.csv").shape
)

fact_orders: (99441, 11)
fact_order_items: (112650, 9)
fact_payments: (99440, 4)
fact_reviews: (98673, 3)
